# Programming Assignment: Categorical Feature Encoding

Welcome to the Categorical Feature Encoding Programming Assignment!

Most real-world datasets contain categorical variables — product categories, customer segments, geographic regions. Machine learning models require numeric input, so choosing the right encoding strategy is critical: the wrong choice can inject false orderings, cause memory issues, or silently leak target information.

**What You Will Do in this Assignment:**

* **One-Hot Encoding** - Apply OHE with `pd.get_dummies` and `OneHotEncoder`, and handle high-cardinality and unseen categories
* **Ordinal Encoding** - Encode ordered categorical features with an explicit, meaningful rank order
* **Frequency Encoding** - Replace categories with their training-set frequency proportions
* **Target Encoding with Smoothing** - Encode categories using target statistics while preventing leakage with Bayesian smoothing
* **Encoding Pipeline** - Build a complete `ColumnTransformer` pipeline applying different encodings to different columns
* **One-Hot Encoder from Scratch** - Implement one-hot encoding using only NumPy
* **Target Encoder from Scratch** - Implement smoothed target encoding using only NumPy and dictionary lookups

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

- [Exercise 1 – One-Hot Encoding](#exercise-1)
- [Exercise 2 – Ordinal Encoding](#exercise-2)
- [Exercise 3 – Frequency Encoding](#exercise-3)
- [Exercise 4 – Target Encoding with Smoothing](#exercise-4)
- [Exercise 5 – Encoding Pipeline](#exercise-5)
- [Exercise 6 – One-Hot Encoder from Scratch](#exercise-6)
- [Exercise 7 – Target Encoder from Scratch](#exercise-7)

## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Dataset

We use a synthetic e-commerce/survey dataset with 500 customers. Each row represents one customer:

| Column | Type | Description |
|:-------|:-----|:------------|
| `product_category` | Nominal | Product type purchased |
| `education_level` | Ordinal | Highest education achieved |
| `satisfaction` | Ordinal | Customer satisfaction rating |
| `city` | Nominal (high-cardinality) | Customer's city (50 unique values) |
| `price` | Numeric | Price paid |
| `age` | Numeric | Customer age |
| `churn` | Binary target | 1 if customer churned, 0 otherwise |

In [ ]:
np.random.seed(42)
N = 500
CATEGORIES   = ['Electronics', 'Clothing', 'Food', 'Books', 'Sports', 'Home']
EDUCATION    = ['High School', 'Bachelor', 'Master', 'PhD']
SATISFACTION = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']

df = pd.DataFrame({
    'product_category': np.random.choice(CATEGORIES, N),
    'education_level':  np.random.choice(EDUCATION, N, p=[0.30, 0.40, 0.20, 0.10]),
    'satisfaction':     np.random.choice(SATISFACTION, N, p=[0.05, 0.15, 0.35, 0.30, 0.15]),
    'city':             np.random.choice([f'City_{i:02d}' for i in range(50)], N),
    'price':            np.random.uniform(5.0, 500.0, N).round(2),
    'age':              np.random.randint(18, 70, N),
})

sat_weight = df['satisfaction'].map({'Poor':3,'Fair':2,'Good':1,'Very Good':0,'Excellent':-1})
edu_weight = df['education_level'].map({'High School':1,'Bachelor':0.5,'Master':0.0,'PhD':-0.5})
logit = -0.5 + 0.4*sat_weight + 0.2*edu_weight + 0.001*df['price'] + np.random.normal(0,1,N)
df['churn'] = (logit > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('churn', axis=1), df['churn'], test_size=0.2, random_state=42)

print(f'Dataset shape: {df.shape}')
print(f'Churn rate:    {df["churn"].mean():.2f}')
print(f'\nFirst 3 rows:')
df.head(3)

<a id='exercise-1'></a>
## Exercise 1 – One-Hot Encoding

### Background

**One-Hot Encoding (OHE)** is the standard technique for **nominal** (unordered) categorical features.  
Each unique category becomes its own binary column: `1` when the category is present, `0` otherwise.

| Before OHE | After OHE |
|:-----------|:----------|
| `product_category = 'Food'` | `category_Food = 1`, all others = 0 |

**When to use:** linear models, logistic regression, SVMs — any model that assumes numeric inputs with no implicit ordering.

### Task

Implement `apply_one_hot_encoding(df, columns)` that:
1. Applies OHE to every column listed in `columns`
2. **Drops** the original categorical columns
3. Returns the modified DataFrame

**Hints:**
- `pd.get_dummies(df, columns=columns, dtype=int)` does all of this in one call.
- Make sure the output column values are integers (0 or 1), not booleans.

In [ ]:
def apply_one_hot_encoding(df, columns):
    ### START CODE HERE ### (≈ 5 lines)
    encoder = None       # OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    X_enc   = None       # fit_transform on df[columns]
    cols    = None       # encoder.get_feature_names_out(columns)
    df_enc  = None       # pd.DataFrame(X_enc, columns=cols, index=df.index)
    return pd.concat([df.drop(columns=columns), df_enc], axis=1)
    ### END CODE HERE ###

In [ ]:
# Quick sanity check
df_ohe = apply_one_hot_encoding(df.drop('churn', axis=1), ['product_category'])
new_cols = [c for c in df_ohe.columns if c.startswith('product_category_')]
print(f'Original columns: {list(df.columns)}')
print(f'New OHE columns:  {new_cols}')
print(f'Rows sum to 1:    {df_ohe[new_cols].sum(axis=1).eq(1).all()}')
df_ohe.head(3)

In [ ]:
unittests.exercise_1(apply_one_hot_encoding)

<a id='exercise-2'></a>
## Exercise 2 – Ordinal Encoding

### Background

**Ordinal encoding** maps each category to an integer that preserves the meaningful **ranking** of the categories.  
It is only appropriate for **ordinal** (ordered) features:

| Satisfaction | Ordinal value |
|:-------------|:--------------|
| Poor         | 0             |
| Fair         | 1             |
| Good         | 2             |
| Very Good    | 3             |
| Excellent    | 4             |

> ⚠️ **Never** apply ordinal encoding to nominal features — it injects false ordering information.

### Task

Implement `apply_ordinal_encoding(df, col, order)` that:
1. Maps each value in `df[col]` to its integer position in the `order` list (0-indexed)
2. Adds the result as a new column named `col + '_encoded'`
3. Keeps the original column unchanged
4. Returns the modified DataFrame

**Hints:**
- Build a `mapping = {cat: i for i, cat in enumerate(order)}`
- Use `Series.map(mapping)` to apply it.

In [ ]:
def apply_ordinal_encoding(df, col, order):
    ### START CODE HERE ### (≈ 4 lines)
    encoder = None       # OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1)
    df = df.copy()
    df[col] = None       # encoder.fit_transform(df[[col]])
    return df
    ### END CODE HERE ###

In [ ]:
EDUCATION_ORDER = ['High School', 'Bachelor', 'Master', 'PhD']
SATISFACTION_ORDER = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']

df_ord = apply_ordinal_encoding(df, 'satisfaction', SATISFACTION_ORDER)
print('Ordinal mapping verification:')
print(df_ord[['satisfaction', 'satisfaction_encoded']].drop_duplicates().sort_values('satisfaction_encoded').to_string())

In [ ]:
unittests.exercise_2(apply_ordinal_encoding)

<a id='exercise-3'></a>
## Exercise 3 – Frequency Encoding

### Background

**Frequency encoding** replaces each category with the **proportion** of rows that contain that category in the training data.

| City    | Count | Frequency |
|:--------|:------|:----------|
| NYC     | 12    | 12/50 = 0.24 |
| LA      | 8     | 8/50 = 0.16 |
| Chicago | 3     | 3/50 = 0.06 |

It produces a **single numeric column** regardless of cardinality — ideal for high-cardinality features and tree-based models.

> ⚠️ Compute frequencies on training data only — never on the full dataset including test rows.

### Task

Implement `apply_frequency_encoding(df, col)` that:
1. Computes the proportion of each category using `value_counts(normalize=True)`
2. Adds a new column `col + '_freq'` with the frequency value for each row
3. Keeps the original column unchanged
4. Returns the modified DataFrame

**Hints:**
- `df[col].value_counts(normalize=True)` returns a Series of proportions.
- Use `.map()` to assign each row its category's proportion.

In [ ]:
def apply_frequency_encoding(df, col):
    ### START CODE HERE ### (≈ 4 lines)
    freq_map = None      # pandas value_counts(normalize=True).to_dict()
    df = df.copy()
    df[col + '_freq'] = None   # map using freq_map
    return df
    ### END CODE HERE ###

In [ ]:
df_freq = apply_frequency_encoding(df, 'city')
print('Top 5 cities by frequency:')
print(df_freq[['city', 'city_freq']].drop_duplicates().sort_values('city_freq', ascending=False).head().to_string())

In [ ]:
unittests.exercise_3(apply_frequency_encoding)

<a id='exercise-4'></a>
## Exercise 4 – Target Encoding with Smoothing

### Background

**Target encoding** replaces each category with the **mean target value** for rows in that category.
Raw means are noisy for rare categories (e.g. a city seen only once in training).
**Smoothing** pulls rare-category estimates towards the global mean:

$$\text{encoded}(c) = \frac{n_c \cdot \bar{y}_c + \alpha \cdot \bar{y}_{\text{global}}}{n_c + \alpha}$$

where $n_c$ = number of rows with category $c$, $\bar{y}_c$ = category mean, $\alpha$ = smoothing strength.

> ⚠️ Always compute target encoding statistics **only from the training set** to prevent target leakage.

### Task

Implement `apply_target_encoding(X_train, y_train, X_test, col, alpha=10)` that:
1. Computes the global mean and per-category (mean, count) from `X_train` + `y_train`
2. Applies the smoothing formula to get `smoothed` per-category encoding
3. Adds `col + '_target_enc'` to copies of both `X_train` and `X_test`
4. For unknown categories in `X_test`, uses the global mean
5. Returns `(X_train_enc, X_test_enc)`

**Hints:**
- Use `pd.DataFrame({'cat': X_train[col], 'y': y_train}).groupby('cat')['y'].agg(['mean','count'])`
- Apply `.fillna(global_mean)` on the test column mapping to handle unknowns.

In [ ]:
def apply_target_encoding(X_train, y_train, X_test, col, alpha=10):
    ### START CODE HERE ### (≈ 7 lines)
    global_mean = None        # np.mean(y_train)
    cat_means   = {}          # dict: category → mean target on training rows
    smoothed    = {}          # apply smoothing formula per category
    X_train = X_train.copy()
    X_test  = X_test.copy()
    X_train[col + '_te'] = None   # map training col using smoothed dict
    X_test[col  + '_te'] = None   # map test col (unseen → global_mean)
    return X_train, X_test
    ### END CODE HERE ###

In [ ]:
X_tr_enc, X_te_enc = apply_target_encoding(X_train, y_train, X_test, 'city', alpha=10)
print('City target encoding — train sample:')
print(X_tr_enc[['city', 'city_target_enc']].drop_duplicates().sort_values('city_target_enc').head(8).to_string())

In [ ]:
unittests.exercise_4(apply_target_encoding)

<a id='exercise-5'></a>
## Exercise 5 – Encoding Pipeline

### Background

A production-ready encoding strategy uses sklearn's `ColumnTransformer` to apply **different encodings to different columns** simultaneously, then feeds the combined output to a classifier.

```
Input DataFrame
  │
  ├─ nominal_cols  ──► OneHotEncoder  ──┐
  ├─ ordinal_col   ──► OrdinalEncoder ──┼──► combined ──► LogisticRegression
  └─ 'price'       ──► StandardScaler ──┘
```

The `Pipeline` object ensures that encoders are **fitted only on training data** during cross-validation, preventing leakage.

### Task

Implement `create_encoding_pipeline(nominal_cols, ordinal_col, ordinal_order)` that:
1. Creates a `ColumnTransformer` with three transformers:
   - `'ohe'`: `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` on `nominal_cols`
   - `'ord'`: `OrdinalEncoder(categories=[ordinal_order])` on `[ordinal_col]`
   - `'num'`: `StandardScaler()` on `['price']`
2. Returns a `Pipeline([('ct', column_transformer), ('clf', LogisticRegression(max_iter=500))])`

**Hints:**
- `ColumnTransformer` takes a list of `(name, transformer, columns)` triples.
- Wrap `ordinal_col` in a list: `[ordinal_col]`.

In [ ]:
def create_encoding_pipeline(nominal_cols, ordinal_col, ordinal_order):
    ### START CODE HERE ### (≈ 6 lines)
    ohe = None       # OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    oe  = None       # OrdinalEncoder(categories=[ordinal_order])
    ct  = None       # ColumnTransformer with ohe for nominal_cols, oe for [ordinal_col], remainder='passthrough'
    return Pipeline([('preprocessing', ct), ('clf', LogisticRegression(max_iter=1000))])
    ### END CODE HERE ###

In [ ]:
EDUCATION_ORDER = ['High School', 'Bachelor', 'Master', 'PhD']

pipeline = create_encoding_pipeline(
    nominal_cols=['product_category'],
    ordinal_col='education_level',
    ordinal_order=EDUCATION_ORDER,
)

X_pipe = df[['product_category', 'education_level', 'price']]
y_pipe = df['churn']

from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline, X_pipe, y_pipe, cv=5, scoring='accuracy')
print(f'5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
unittests.exercise_5(create_encoding_pipeline)

<a id='exercise-6'></a>
## Exercise 6 – One-Hot Encoder from Scratch

### Background

Understanding how OHE works internally helps you debug edge cases and adapt it to custom requirements.
The algorithm is:

1. Collect all unique categories and sort them.
2. Create a zero matrix of shape `(n_samples, n_unique)`.
3. For each row `i`, set the column corresponding to `series[i]` to 1.

### Task

Implement `one_hot_encode_from_scratch(series)` that:
1. Collects all **unique** values from the Series and sorts them
2. Creates a **zero integer matrix** of shape `(len(series), n_unique)`
3. For each row, sets the correct column to 1
4. Returns `(matrix, categories)` where `categories` is the sorted list of unique values

**Hints:**
- `sorted(series.unique())` gives the sorted categories.
- Build a dict `cat_to_idx = {c: i for i, c in enumerate(categories)}` for fast lookup.
- Use `np.zeros((n, k), dtype=int)` to create the matrix.

In [ ]:
def one_hot_encode_from_scratch(series):
    ### START CODE HERE ### (≈ 6 lines)
    categories = None     # sorted unique values from series
    result     = {}       # build a dict: category → 0/1 array
    for cat in categories:
        result[cat] = None   # (series.values == cat).astype(int)
    return pd.DataFrame(result, index=series.index)
    ### END CODE HERE ###

In [ ]:
demo_series = pd.Series(['Food', 'Electronics', 'Food', 'Books', 'Electronics'])
matrix, cats = one_hot_encode_from_scratch(demo_series)
print(f'Categories (sorted): {cats}')
print(f'Matrix shape:        {matrix.shape}')
print(f'Matrix:\n{matrix}')
print(f'Row sums:            {matrix.sum(axis=1).tolist()}')

In [ ]:
unittests.exercise_6(one_hot_encode_from_scratch)

<a id='exercise-7'></a>
## Exercise 7 – Target Encoder from Scratch

### Background

Now you will implement **smoothed target encoding** from first principles using only NumPy and pandas — no sklearn.

Recall the smoothing formula:

$$\text{encoded}(c) = \frac{n_c \cdot \bar{y}_c + \alpha \cdot \bar{y}_{\text{global}}}{n_c + \alpha}$$

For **unknown categories** at test time (categories not seen during training), fall back to the global mean.

### Task

Implement `target_encode_from_scratch(train_col, y_train, test_col, alpha=10)` that:
1. Computes the global mean of `y_train`
2. Computes per-category mean and count from `train_col` and `y_train`
3. Applies the smoothing formula to get the mapping for each category
4. Encodes `train_col` and `test_col` using this mapping
5. Uses the global mean for categories in `test_col` not seen in training
6. Returns `(train_encoded, test_encoded)` as `np.ndarray`

**Hints:**
- Use `pd.DataFrame({'cat': train_col, 'y': y_train}).groupby('cat')['y'].agg(['mean','count'])`
- Use `mapping.get(c, global_mean)` to handle unknown categories.

In [ ]:
def target_encode_from_scratch(train_col, y_train, test_col, alpha=10):
    ### START CODE HERE ### (≈ 8 lines)
    global_mean = None    # np.mean(y_train)
    enc_map     = {}      # category → smoothed mean
    for cat in train_col.unique():
        mask      = None  # train_col == cat
        n         = None  # mask.sum()
        cat_mean  = None  # y_train[mask].mean()
        enc_map[cat] = None  # smoothing formula: (n*cat_mean + alpha*global_mean) / (n + alpha)
    train_enc = None  # train_col.map(enc_map).fillna(global_mean)
    test_enc  = None  # test_col.map(enc_map).fillna(global_mean)
    return train_enc, test_enc
    ### END CODE HERE ###

In [ ]:
train_cities = X_train['city'].reset_index(drop=True)
test_cities  = X_test['city'].reset_index(drop=True)

tr_enc, te_enc = target_encode_from_scratch(
    train_cities, y_train.values, test_cities, alpha=10
)
print(f'Train encoding — first 5: {tr_enc[:5].round(3).tolist()}')
print(f'Test  encoding — first 5: {te_enc[:5].round(3).tolist()}')
print(f'Global mean: {y_train.mean():.4f}')

In [ ]:
unittests.exercise_7(target_encode_from_scratch)